# Burn a landscape with pyretechnics

The minimal `fb_tools` path from a LANDFIRE landscape to a spread realization: de-stack an LCP,
pick a real weather episode, give it spatially resolved HRRR wind, ignite, burn, look. Runs in the
`fb_pyre` env. Project work (arms, design fires, paired deltas) builds on these same calls — see the
COSWAP notebook `05_PyreTransmission.ipynb` in that project folder.

Landscape: the `front_range` test AOI (pyrome 46), downloaded once with `lfps_request` into
`data/spatial/mod/fuelscapes/front_range/`. Weather: pyrome-mean RTMA (T/RH → fuel moisture) and
HRRR 10 m wind over the domain, both from the caches under `data/weather/`.


In [ ]:
import json, os, time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio, rasterio.plot
from shapely.geometry import Point

import fb_tools
from fb_tools.models import pyre
from fb_tools.spread import (destack_lcp, select_episodes, load_rtma_hourly, write_episode_frames,
                             layer_bounds_lonlat, episode_wind_cube, write_wind_cube, load_wind_cube,
                             wind_cube_summary, weather_arrays, summarize_realization, weather_from_scenario)
from fb_tools.weather.hrrr import fetch_hrrr_wind_grid

REPO     = Path(fb_tools.__file__).resolve().parents[1]
LCP_FP   = next((REPO / "data/spatial/mod/fuelscapes/front_range").glob("LCP_front_range_LF*.tif"))
SCRATCH  = Path(os.environ.get("FB_PYRE_SCRATCH", Path.home() / "fb_scratch")) / "pyre_example"
ERC_DIR  = REPO / "data/weather/pyrome_erc"
RTMA_DIR = REPO / "data/weather/pyrome_emc/RTMA_MONTHLY"
WIND_DIR = REPO / "data/weather/pyrome_emc/RTMA_WIND_MONTHLY"

PYROME        = 46
SEVERITY      = "p97"          # ERC percentile that day 1 of the episode must meet
DURATION_DAYS = 7
CROWN_GATE    = 0.0            # canopy-cover gate for crown fire (0.0 = off; stock engine 0.4)
SEED          = 617327

pyre.pyre_version_info()


## 1. Static layers

`destack_lcp` writes one GeoTIFF per engine input in engine units (slope as rise/run, canopy as fractions).

In [ ]:
layers = SCRATCH / "layers" / LCP_FP.stem
if not (layers / "fuel_model.tif").exists():
    destack_lcp(LCP_FP, layers)
static, TRANSFORM, CRS = pyre.load_static_layers(layers)
SHAPE = static["slope"].shape
print(SHAPE, CRS, f"{abs(TRANSFORM.a):.0f} m", "| burnable fraction:", round(float(np.isin(static["fuel_model"], [0, 91, 92, 93, 98, 99], invert=True).mean()), 3))


## 2. Potential fire behavior (no spread)

A FlamMap-style map from the engine's `burn_cells`: every cell burned as a head fire under one weather state.

In [ ]:
p97 = json.load(open(REPO / f"data/weather/flammap_rtma/pyrome_{PYROME}_flammap.json"))["scenarios"]["p97"]
scen = weather_from_scenario(p97, wind_from_deg=270.0, tmp_c=30.0)
pot = pyre.burn_landscape(layers, scen, SCRATCH / "potential_p97", crown_gate=CROWN_GATE)
fig, axes = plt.subplots(1, 2, figsize=(5, 4))
for ax, layer, kw in zip(axes, ["flame_length", "fire_type"], [dict(cmap="inferno", vmin=0, vmax=8), dict(cmap="viridis", vmin=0, vmax=3)]):
    with rasterio.open(SCRATCH / "potential_p97" / f"{layer}.tif") as src:
        im = ax.imshow(src.read(1), extent=rasterio.plot.plotting_extent(src), **kw)
    plt.colorbar(im, ax=ax, shrink=0.7, label=layer); ax.set_axis_off()
plt.show(); {k: pot[k] for k in ("n_burnable", "crown_frac", "wall_s")}


## 3. A weather episode

A real historical window whose day-1 ERC meets the severity percentile. T/RH come from the pyrome-mean RTMA record and are turned into NFDRS dead fuel moisture; the frame is in engine units.

In [ ]:
rtma = load_rtma_hourly(RTMA_DIR, PYROME, range(2016, 2026), wind_dir=WIND_DIR)
episodes = select_episodes(PYROME, ERC_DIR, severity=SEVERITY, n=1, duration_days=DURATION_DAYS,
                           years=range(2016, 2026), rtma_df=rtma, seed=SEED)
frame_csv = write_episode_frames(episodes, rtma, SCRATCH / "episodes")["ep_000"]
frame = pd.read_csv(frame_csv, parse_dates=["datetime_utc"])
episodes[["episode_id", "start_date", "end_date", "erc_day1"]]


### 3b. Local wind

Any spatial mean of wind erases the gusts that drive spread: the pyrome-mean wind rarely exceeds
17 mph while single 3 km HRRR cells in the same domain reach 30. So the episode's wind is a 3-D cube:
HRRR 10 m U/V fetched by byte range from AWS (~2 MB per hour, cached), sampled nearest-cell onto a
block grid that divides the landscape exactly. The engine repeats each block over its 30 m cells.

In [ ]:
cube_fp = SCRATCH / "episodes" / "ep_000_wind.npz"
if not cube_fp.exists():
    grid = fetch_hrrr_wind_grid(frame.datetime_utc, layer_bounds_lonlat(layers), SCRATCH / "hrrr_cache", n_threads=6)
    write_wind_cube(cube_fp, episode_wind_cube(grid, TRANSFORM, CRS, SHAPE))
cube = load_wind_cube(cube_fp)
s = wind_cube_summary(cube)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.fill_between(s.hour, s.ws_mean, s.ws_max, color="C1", alpha=0.3, label="HRRR domain mean → max cell")
ax.plot(frame.hour, frame.ws_kmh, color="k", lw=0.8, label="RTMA pyrome mean")
ax.set(xlabel="hour since ignition", ylabel="km/h"); ax.legend(); plt.show()
print("cube", cube["wind_speed_10m"].shape, "| block", cube["block"], "cells =", cube["block_m"], "m")


## 4. Ignite and burn

A 10-acre circle at the landscape centre. `run_realization` writes arrival time, fire type, spread rate/direction, intensity and flame length as GeoTIFFs plus a manifest recording the seed, engine build, gate and solver knobs.

In [ ]:
cx, cy = TRANSFORM * (SHAPE[1] / 2, SHAPE[0] / 2)
ign_fp = SCRATCH / "ignition.shp"
gpd.GeoDataFrame(geometry=[Point(cx, cy).buffer(np.sqrt(10 * 4046.86 / np.pi))], crs=CRS).to_file(ign_fp)

run_dir = SCRATCH / "runs" / "ep_000"
t0 = time.perf_counter()
man = pyre.run_realization(layers, weather_arrays(frame, wind_cube=cube_fp), ign_fp, run_dir,
                           seed=SEED, duration_h=DURATION_DAYS * 24, spot_config=pyre.SPOT_CENTRAL,
                           crown_gate=CROWN_GATE, overwrite=True)
print(f"{time.perf_counter() - t0:.0f} s | burned {man['burned_ha']:,.0f} ha | {man['stop_condition']}")
{k: man[k] for k in ("seed", "crown_gate", "wind_cube_shape", "pending_spot_cells", "pyretechnics_version")}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
with rasterio.open(run_dir / "time_of_arrival.tif") as src:
    im = axes[0].imshow(src.read(1) / 1440.0, extent=rasterio.plot.plotting_extent(src), cmap="magma_r", vmin=0, vmax=DURATION_DAYS)
plt.colorbar(im, ax=axes[0], shrink=0.7, label="arrival (days)")
with rasterio.open(run_dir / "flame_length.tif") as src:
    im = axes[1].imshow(src.read(1), extent=rasterio.plot.plotting_extent(src), cmap="inferno", vmin=0, vmax=8)
plt.colorbar(im, ax=axes[1], shrink=0.7, label="flame length (m)")
for ax in axes:
    gpd.read_file(ign_fp).plot(ax=ax, color="cyan"); ax.set_axis_off()
plt.show()
pd.Series(summarize_realization(run_dir))

## Next

- Several episodes at once: `pyre.build_runs(arms, ignitions, episodes, conditions, wind_cubes=...)` then
  `pyre.run_batch(...)`; `aggregate_realizations` turns a set of runs into burn probability and mean
  arrival rasters.
- Counterfactuals: add a second `arms` entry (a treated LCP) — same seeds, same weather, exact pairing.
  The COSWAP project notebook is the worked example.
